# Titanic Model

In [168]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("train.csv")
df.head(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


In [169]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [170]:
# the only difference from a regular function is that we must input a dataframe (2D) and output a dataframe (2D)

def name_to_title(df: pd.DataFrame):
    name_col = df["Name"]
    name_col = name_col.apply(lambda name: name[name.index(", ") + 2:name.index(".")])
    
    titles = {
        "Mr": "Mr",
        "Mrs": "Mrs",
        "Miss": "Miss",
        "Master": "Master"
    }

    name_col = name_col.apply(lambda name: titles.get(name, "Rare"))
    
    return name_col.to_frame(name="Title")

def sex_to_num(df: pd.DataFrame):
    result = df.copy()
    result["Sex"] = result["Sex"].map({
        "female": 1.0,
        "male": 0.0,
    })
    return result

def has_cabin(df: pd.DataFrame):
    df["HasCabin"] = df["Cabin"].notna().astype(float)

    return df["HasCabin"].to_frame()

In [199]:
from sklearn.base import BaseEstimator, TransformerMixin

class AgeImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_cols, target_col):
        self.group_cols = group_cols
        self.target_col = target_col

    def fit(self, X: pd.DataFrame, y=None) -> "AgeImputer":
        X = X.copy()

        self.medians_ = (
            X.groupby(self.group_cols, dropna=False)[self.target_col].median()
        )

        self.global_median_ = X[self.target_col].median()

        return self
    
    def transform(self, X: pd.DataFrame):
        X = X.copy()

        missing_mask = X[self.target_col].isna()

        if len(self.group_cols) == 1:
            group_medians = X[self.group_cols[0]].map(self.medians_)
        else:
            group_keys = pd.MultiIndex.from_frame(X[self.group_cols])

            group_medians = pd.Series(
                self.medians_.reindex(group_keys).to_numpy(),
                index=X.index
            )
        
        X.loc[missing_mask, self.target_col] = (
            group_medians.loc[missing_mask].fillna(self.global_median_)
        )

        return X[["Age"]]

    def get_feature_names_out(self, input_features=None):
        return np.asarray([self.target_col], dtype=object)

In [172]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [173]:
ai = AgeImputer(["Pclass", "Sex"], "Age")
ai.fit(df)
ai.transform(df)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,21.5,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [200]:
name_pipe = Pipeline([
    ("extract_title", FunctionTransformer(
        func=name_to_title,
        feature_names_out=lambda _, features: ["Title"]
    )),
    ("encoder", OneHotEncoder())
])

gen_pipe = Pipeline([
    ("map_sex", FunctionTransformer(
        func=sex_to_num,
        feature_names_out="one-to-one"
    ))
])

hot_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder())
])

age_pipe = Pipeline([
    ("age-impute", AgeImputer(
        group_cols=["Pclass", "Sex"],
        target_col="Age"
    ))
])

cabin_pipe = Pipeline([
    ("has-cabin", FunctionTransformer(
        func=has_cabin,
        feature_names_out=lambda _, features: ["Has Cabin"]
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        (
            "title",
            name_pipe,
            ["Name"]
        ),
        (
            "gender",
            gen_pipe,
            ["Sex"]
        ),
        (
            "age",
            age_pipe,
            ["Age", "Pclass", "Sex"]
        ),
        (
            "one_hot",
            hot_pipe,
            ["Embarked"]
        ),
        (
            'drop_cols',
            'drop',
            ["Ticket", "PassengerId", "Cabin"]
        ),
    ],
    remainder="passthrough"
)


In [208]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

fitted_preprocess = preprocess.fit_transform(X)

print("Missing values:", np.sum(np.isnan(fitted_preprocess)))
pd.DataFrame(fitted_preprocess)

Missing values: 0


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.0,0.0,1.0,0.0,0.0,0.0,22.0,0.0,0.0,1.0,1.0,0.0,7.2500
1,0.0,0.0,0.0,1.0,0.0,1.0,38.0,1.0,0.0,0.0,1.0,0.0,71.2833
2,0.0,1.0,0.0,0.0,0.0,1.0,26.0,0.0,0.0,1.0,0.0,0.0,7.9250
3,0.0,0.0,0.0,1.0,0.0,1.0,35.0,0.0,0.0,1.0,1.0,0.0,53.1000
4,0.0,0.0,1.0,0.0,0.0,0.0,35.0,0.0,0.0,1.0,0.0,0.0,8.0500
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,0.0,0.0,0.0,1.0,0.0,27.0,0.0,0.0,1.0,0.0,0.0,13.0000
887,0.0,1.0,0.0,0.0,0.0,1.0,19.0,0.0,0.0,1.0,0.0,0.0,30.0000
888,0.0,1.0,0.0,0.0,0.0,1.0,21.5,0.0,0.0,1.0,1.0,2.0,23.4500
889,0.0,0.0,1.0,0.0,0.0,0.0,26.0,1.0,0.0,0.0,0.0,0.0,30.0000
